# CAD 01 -- Bezier Curves & NURBS: The Math Behind Every CAD Tool

> **Geo-Instructor · CAD/Parametric Career Track · Notebook 1 of 3**

---

Every surface in Rhino, every path in Illustrator, every body in STEP files is a NURBS curve or surface.
This notebook builds the math from scratch: Bezier first, then B-splines, then NURBS.

```
Runtime: ~3 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace','axes.grid':True,'grid.alpha':0.3})
print('Ready.')

---
## Part 1 -- De Casteljau: Geometric Subdivision

Given control points P0..Pn and parameter t in [0,1], the De Casteljau algorithm
recursively linearly interpolates until one point remains. That point is on the curve.

```
  r=0:  P0    P1    P2    P3
  r=1:   Q01   Q12   Q23
  r=2:     R012  R123
  r=3:       S0123  <-- point on curve at t
```

In [ ]:
def de_casteljau(ctrl_pts, t):
    """Evaluate Bezier curve at t using De Casteljau. Returns (point, pyramid)."""
    pts = [p.copy() for p in ctrl_pts]
    pyramid = [pts[:]]
    for r in range(1, len(ctrl_pts)):
        new_pts = [(1-t)*pts[i] + t*pts[i+1] for i in range(len(pts)-1)]
        pts = new_pts
        pyramid.append(pts[:])
    return pts[0], pyramid

# Cubic Bezier control points
P = [np.array([0.0,0.0]), np.array([1.0,2.0]), np.array([3.0,2.0]), np.array([4.0,0.0])]
t_val = 0.4
pt, pyramid = de_casteljau(P, t_val)

# Evaluate full curve
ts = np.linspace(0, 1, 200)
curve = np.array([de_casteljau(P, t)[0] for t in ts])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('De Casteljau Algorithm -- Geometric Construction', fontsize=11)

colors = ['steelblue','tomato','seagreen','gold']
ax = axes[0]
for i, level in enumerate(pyramid):
    pts_arr = np.array(level)
    ax.plot(pts_arr[:,0], pts_arr[:,1], 'o-', color=colors[i], lw=1.5, ms=7, label=f'Level {i}')
ax.plot(curve[:,0], curve[:,1], 'k-', lw=2.5, label='Bezier curve')
ax.plot(*pt, 'r*', ms=15, label=f'Point at t={t_val}')
ctrl = np.array(P)
ax.plot(ctrl[:,0], ctrl[:,1], 's--', color='gray', lw=1, ms=8, label='Control polygon', zorder=1)
ax.legend(fontsize=8); ax.set_title(f'De Casteljau at t={t_val}'); ax.set_aspect('equal')

ax2 = axes[1]
for t_demo in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pt_d, pyr_d = de_casteljau(P, t_demo)
    ax2.plot(*pt_d, 'o', color='tomato', ms=8, zorder=5)
ax2.plot(curve[:,0], curve[:,1], 'steelblue', lw=2.5)
ax2.plot(ctrl[:,0], ctrl[:,1], 's--', color='gray', lw=1, ms=8)

# Convex hull
from scipy.spatial import ConvexHull
hull = ConvexHull(np.array(P))
hull_pts = np.array(P)[hull.vertices]
hull_pts = np.vstack([hull_pts, hull_pts[0]])
ax2.fill(hull_pts[:,0], hull_pts[:,1], alpha=0.08, color='steelblue')
ax2.plot(hull_pts[:,0], hull_pts[:,1], 'steelblue', lw=1, ls='--', alpha=0.5, label='Convex hull')
ax2.set_aspect('equal'); ax2.legend(fontsize=8)
ax2.set_title('Curve stays inside convex hull of control points')
plt.tight_layout(); plt.show()
print('Convex hull property: the curve is always inside the hull of its control points.')

---
## Part 2 -- Bernstein Basis Polynomials

The Bezier curve is a weighted sum of Bernstein polynomials:
```
  B(t) = sum_i  B_{i,n}(t) * P_i
  B_{i,n}(t) = C(n,i) * t^i * (1-t)^(n-i)
```
They are non-negative, sum to 1 (partition of unity), and each peaks at t=i/n.

In [ ]:
from math import comb

def bernstein(i, n, t):
    return comb(n, i) * (t**i) * ((1-t)**(n-i))

ts = np.linspace(0, 1, 300)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, [2, 3, 5]):
    for i in range(n+1):
        b = np.array([bernstein(i, n, t) for t in ts])
        ax.plot(ts, b, lw=2, label=f'B_{i},{n}')
    ax.set_title(f'Degree {n} Bernstein basis')
    ax.set_xlabel('t'); ax.set_ylabel('B_i,n(t)')
    ax.legend(fontsize=7)
    # Verify partition of unity
    total = sum(bernstein(i, n, 0.5) for i in range(n+1))
    ax.text(0.5, 1.02, f'sum at t=0.5: {total:.6f}', transform=ax.transAxes, ha='center', fontsize=8, color='gray')
plt.tight_layout(); plt.show()
print('Bernstein polynomials sum to 1 at every t -- partition of unity guaranteed.')

In [ ]:
# Compare curves of different degree
base_ctrl = [np.array([0.,0.]), np.array([0.5,2.]), np.array([2.,2.5]),
             np.array([3.5,1.5]), np.array([4.,0.]), np.array([3.,-1.])]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Bezier curves -- degree comparison', fontsize=11)
for ax, n_ctrl in zip(axes, [3, 4, 6]):
    ctrl = base_ctrl[:n_ctrl]
    ts = np.linspace(0, 1, 300)
    curve = np.array([de_casteljau(ctrl, t)[0] for t in ts])
    ctrl_arr = np.array(ctrl)
    ax.plot(ctrl_arr[:,0], ctrl_arr[:,1], 'o--', color='gray', lw=1, ms=8)
    ax.plot(curve[:,0], curve[:,1], 'steelblue', lw=2.5)
    for j, p in enumerate(ctrl):
        ax.annotate(f'P{j}', p, textcoords='offset points', xytext=(5,5), fontsize=8)
    ax.set_title(f'Degree {n_ctrl-1} ({n_ctrl} control pts)')
    ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Higher degree = smoother but global control (moving 1 point affects whole curve).')

---
## Part 3 -- B-Splines: Local Control via Knot Vectors

The core problem with Bezier: moving ONE control point changes the ENTIRE curve.
B-splines fix this with a **knot vector** that gives each basis function a finite support.

```
  B-spline:  C(t) = sum_i  N_{i,k}(t) * P_i
  N_{i,1}(t) = 1 if t in [t_i, t_{i+1}), else 0
  N_{i,k}(t) = (t-t_i)/(t_{i+k-1}-t_i) * N_{i,k-1}(t)
             + (t_{i+k}-t)/(t_{i+k}-t_{i+1}) * N_{i+1,k-1}(t)
```
This is the **Cox-de Boor recursion**.

In [ ]:
def bspline_basis(i, k, t, knots):
    """Cox-de Boor recursion for B-spline basis N_{i,k}(t)."""
    if k == 1:
        if knots[i] <= t < knots[i+1]: return 1.0
        if t == knots[-1] and knots[i] <= t <= knots[i+1]: return 1.0
        return 0.0
    denom1 = knots[i+k-1] - knots[i]
    denom2 = knots[i+k]   - knots[i+1]
    c1 = ((t - knots[i]) / denom1 * bspline_basis(i, k-1, t, knots)) if denom1 > 0 else 0.0
    c2 = ((knots[i+k] - t) / denom2 * bspline_basis(i+1, k-1, t, knots)) if denom2 > 0 else 0.0
    return c1 + c2

def eval_bspline(ctrl_pts, knots, k, t):
    n = len(ctrl_pts)
    pt = np.zeros(2)
    for i in range(n):
        pt += bspline_basis(i, k, t, knots) * ctrl_pts[i]
    return pt

# Uniform open knot vector for 6 ctrl points, degree 3
n_ctrl = 6; k = 4  # degree = k-1 = 3
knots_open   = [0,0,0,0, 1,2, 3,3,3,3]  # clamped
knots_uniform = [0,1,2,3,4,5,6,7,8,9]   # unclamped

ts = np.linspace(knots_open[k-1], knots_open[-(k)], 400)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('B-Spline Basis Functions (Cox-de Boor)', fontsize=11)
for ax, (knots, label) in zip(axes, [(knots_open,'Clamped (open)'), (knots_uniform,'Uniform (unclamped)')]):
    ts_b = np.linspace(knots[k-1], knots[-(k)]+1e-10, 300)
    for i in range(n_ctrl):
        basis = [bspline_basis(i, k, t, knots) for t in ts_b]
        ax.plot(ts_b, basis, lw=2, label=f'N_{i},k')
    total = [sum(bspline_basis(i,k,t,knots) for i in range(n_ctrl)) for t in ts_b]
    ax.plot(ts_b, total, 'k--', lw=1.5, alpha=0.5, label='sum=1')
    ax.set_title(label); ax.legend(fontsize=7); ax.set_xlabel('t')
plt.tight_layout(); plt.show()
print('Local support: each basis function is nonzero over at most k knot spans.')

In [ ]:
# Demonstrate local control: move one control point
ctrl = [np.array([0.,0.]), np.array([1.,1.5]), np.array([2.,2.]),
        np.array([3.,1.5]), np.array([4.,2.5]), np.array([5.,0.])]
knots = [0,0,0,0, 1,2, 3,3,3,3]; k = 4
ts = np.linspace(0, 3, 400)

ctrl_moved = [p.copy() for p in ctrl]
ctrl_moved[2] = np.array([2., 3.8])  # move only P2

curve_orig  = np.array([eval_bspline(ctrl, knots, k, t) for t in ts])
curve_moved = np.array([eval_bspline(ctrl_moved, knots, k, t) for t in ts])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (crv, cpts, title) in zip(axes, [
    (curve_orig, ctrl,  'Original B-Spline'),
    (curve_moved,ctrl_moved,'P2 moved up -- only local region changes')]):
    carr = np.array(cpts)
    ax.plot(carr[:,0], carr[:,1], 'o--', color='gray', lw=1, ms=8)
    ax.plot(crv[:,0], crv[:,1], 'steelblue', lw=2.5)
    for j, p in enumerate(cpts):
        ax.annotate(f'P{j}', p, textcoords='offset points', xytext=(5,3), fontsize=8,
                    color='tomato' if j==2 else 'k')
    ax.set_title(title); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('Moving P2 only changes the curve near P2 -- local control preserved.')

---
## Part 4 -- NURBS: Adding Rational Weights for Exact Conics

B-splines cannot represent circles exactly. NURBS (Non-Uniform Rational B-Splines)
add a weight w_i to each control point:
```
  C(t) = sum_i  (N_{i,k}(t) * w_i * P_i)  /  sum_j (N_{j,k}(t) * w_j)
```
With the right weights, a NURBS can represent a **perfect circle**.

In [ ]:
# NURBS circle using 9 control points (quadrant-by-quadrant, degree 2)
# Classic construction: 3 control points per quadrant with w=cos(45deg)=sqrt(2)/2 at corners

w = np.cos(np.pi/4)  # = sqrt(2)/2 ~0.707
ctrl = np.array([
    [1.,  0.], [1.,  1.], [0.,  1.],  # Q1
    [-1., 1.], [-1., 0.],             # Q2 corner
    [-1.,-1.], [0., -1.],             # Q3
    [1., -1.], [1.,  0.],             # Q4
])
weights = np.array([1., w, 1., w, 1., w, 1., w, 1.])
knots   = [0,0,0, 1,1, 2,2, 3,3, 4,4,4]  # degree=2 clamped with repeated interior

def eval_nurbs(ctrl, weights, knots, k, t):
    n = len(ctrl)
    num = np.zeros(ctrl.shape[1])
    den = 0.0
    for i in range(n):
        b = bspline_basis(i, k, t, knots)
        num += b * weights[i] * ctrl[i]
        den += b * weights[i]
    return num / den if den > 1e-12 else num

ts = np.linspace(0, 4, 1000)
curve = np.array([eval_nurbs(ctrl, weights, knots, 3, t) for t in ts])

# Check: are all points at radius 1?
radii = np.linalg.norm(curve, axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
ax1.plot(curve[:,0], curve[:,1], 'steelblue', lw=2.5, label='NURBS circle')
ax1.plot(ctrl[:,0],  ctrl[:,1],  'o--', color='gray', lw=1, ms=8, label='Control polygon')
for i, (p, wt) in enumerate(zip(ctrl, weights)):
    ax1.annotate(f'w={wt:.2f}', p, textcoords='offset points', xytext=(4,3), fontsize=7, color='tomato')
ax1.set_aspect('equal'); ax1.legend(fontsize=8)
ax1.set_title('NURBS circle (degree 2, 9 control points)')
ax1.add_patch(plt.Circle((0,0), 1, fill=False, color='tomato', lw=1, ls='--', alpha=0.5))

ax2.plot(ts, radii, 'steelblue', lw=2)
ax2.axhline(1.0, color='tomato', ls='--', lw=1.5, label='Exact radius = 1')
ax2.set_xlabel('t'); ax2.set_ylabel('||C(t)||')
ax2.set_title(f'Distance from origin\nmax deviation: {np.max(np.abs(radii-1)):.2e}')
ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f'Max radius error: {np.max(np.abs(radii-1)):.2e}  (numerical precision only)')

---
## Exercises

### Exercise 1 -- Bezier curve fitting
Given N data points, find control points that minimize the sum of squared distances from the points to the Bezier curve. This is used in font outline extraction and path simplification.

### Exercise 2 -- G1 continuity
Join two cubic Bezier curves with G1 (tangent) continuity. The rule: the last control point of curve A, the junction point, and the first control point of curve B must be collinear.

### Exercise 3 -- NURBS surface
Extend the curve to a surface by applying the basis in two parameter directions u and v.
Generate a NURBS cylinder or sphere.

---
## Summary

| Concept | Formula | Used in |
|---------|---------|--------|
| De Casteljau | recursive lerp | Bezier evaluation, subdivision |
| Bernstein basis | C(n,i)*t^i*(1-t)^(n-i) | Bezier blending |
| Cox-de Boor | recursive knot formula | B-spline, NURBS evaluation |
| NURBS | rational weighted B-spline | Rhino, Grasshopper, STEP, IGES |
| Knot vector | controls continuity & support | all parametric CAD systems |

In [ ]:
# Final demo: B-spline curve shaped like a Grasshopper logo (abstract)
print('Notebook complete.')
print('Key takeaways:')
print('  De Casteljau  -> geometric, stable, O(n^2) per point')
print('  Bernstein     -> algebraic form, partition of unity')
print('  B-spline      -> local control via knot vector')
print('  NURBS         -> exact conics via rational weights')
print()
print('Production tools: Rhino, FreeCAD, STEP/IGES format, OpenCASCADE kernel')